# Clase 023 — Indexación (loc, iloc, at, iat)

**Parte 0** · VanderPlas cap. 3 § 3.3.

> 🎯 4 indexers, cuándo usar cada uno, evitar SettingWithCopyWarning.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)

# DataFrame de demo
df = pd.DataFrame({
    'nombre': ['Ana', 'Bob', 'Cris', 'Dan', 'Eli'],
    'edad'  : [30, 25, 28, 35, 22],
    'nota'  : [7.5, 6.0, 8.2, 5.8, 9.1],
}, index=['a', 'b', 'c', 'd', 'e'])
print(df)

## 1️⃣ Los 4 indexers

| Indexer | Selector | Inclusivo slicing? | Uso |
|---|---|---|---|
| `df[...]` | label de columna | — | shortcut, devuelve Series |
| `.loc[row, col]` | **label** | **sí** (incluye end) | el del 80% del tiempo |
| `.iloc[row, col]` | **posición** entera | no (Python style) | cuando no importa el label |
| `.at[row, col]` | label, single value | — | rápido, 1 celda |
| `.iat[row, col]` | posición, single value | — | rápido, 1 celda |

## 2️⃣ Acceso por columna — los 3 caminos

In [ ]:
# Atributo: cómodo pero quirky
print('df.nombre:')
print(df.nombre.values)

# Bracket: el más explícito
print('\ndf["nombre"]:')
print(df['nombre'].values)

# .loc: el más rico (permite combinar filas + cols)
print('\ndf.loc[:, "nombre"]:')
print(df.loc[:, 'nombre'].values)

print('\n⚠️ atributo falla si el nombre tiene espacios, choca con métodos (df.shape), o es número.')

## 3️⃣ loc inclusivo vs iloc exclusivo

Esto sorprende a todo el mundo viniendo de Python puro:

In [ ]:
print('df.loc["a":"c"] — INCLUSIVE end (3 filas: a, b, c):')
print(df.loc['a':'c'])
print('\ndf.iloc[0:3] — EXCLUSIVE end (3 filas: posiciones 0, 1, 2):')
print(df.iloc[0:3])

## 4️⃣ Filtro + columnas con loc

In [ ]:
# Filas donde nota > 7, columnas nombre y nota
result = df.loc[df['nota'] > 7, ['nombre', 'nota']]
print(result)

## 5️⃣ `.at` y `.iat` — single value rápido

Útiles cuando estás en un loop y solo quieres una celda — son ~10× más rápidos que `.loc`/`.iloc` para single value.

In [ ]:
import time

# Mucho más rápido para 1 celda
t0 = time.perf_counter()
for _ in range(10_000):
    _ = df.loc['a', 'nota']
t1 = time.perf_counter()

t2 = time.perf_counter()
for _ in range(10_000):
    _ = df.at['a', 'nota']
t3 = time.perf_counter()

print(f'.loc 10k veces: {(t1-t0)*1000:.1f} ms')
print(f'.at  10k veces: {(t3-t2)*1000:.1f} ms')
print(f'speedup       : {(t1-t0)/(t3-t2):.1f}×')

## 6️⃣ ⚠️ `SettingWithCopyWarning`

El bug más confuso de pandas. Ocurre cuando asignas a una **vista** y pandas no sabe si afectará al original:

```python
subset = df[df['edad'] > 25]   # ¿vista o copia?
subset['nuevo'] = 1            # SettingWithCopyWarning
```

**Fix**: usa `.loc` para hacer la asignación en una sola expresión:

```python
df.loc[df['edad'] > 25, 'nuevo'] = 1   # sin warning
```

En pandas 3+ esto será error duro, no warning.

In [ ]:
import warnings

# Provoca el warning
df_copy = df.copy()
with warnings.catch_warnings():
    warnings.simplefilter('always')
    try:
        subset = df_copy[df_copy['edad'] > 25]
        subset['nuevo'] = 1
    except Exception as e:
        print(f'En pandas modernos puede ser error: {e}')

# Forma correcta
df_copy.loc[df_copy['edad'] > 25, 'nuevo'] = 1
print(df_copy)

## ✅ Checklist

- [ ] Sé los 4 indexers (loc, iloc, at, iat) y cuándo cada uno
- [ ] Recuerdo que `.loc` slicing es inclusivo, `.iloc` exclusivo
- [ ] Uso `.loc[mask, cols]` para filtrar y seleccionar
- [ ] Asigno con `.loc` para evitar SettingWithCopyWarning
- [ ] Uso `.at`/`.iat` cuando estoy en loops

## 📝 Homework

Ver `README.md`. 3 métodos acceso, loc vs iloc, filtro complejo, SettingWithCopyWarning.

## 📖 Definiciones y características

**`.loc[fila, col]`**

Acceso por **etiqueta**. Slicing es **inclusivo** en ambos extremos (`df.loc['a':'c']` incluye 'c'). Acepta booleano: `df.loc[df['x'] > 0, 'col']`.

**`.iloc[fila, col]`**

Acceso por **posición entera**. Slicing es **exclusivo** del extremo (estilo Python). `df.iloc[0:5]` da 5 filas (índices 0..4).

**`.at[fila, col]` / `.iat[fila, col]`**

Versiones para acceder/asignar **un único valor**. ~10× más rápidos que `.loc`/`.iloc` cuando estás en loops. Mismo patrón label vs posición.

**Indexer chaining (`df[cond][col] = x`)**

Encadenar dos `[]` operaciones de acceso. Crea ambigüedad: ¿es vista o copia? Origen del `SettingWithCopyWarning`. **Evítalo siempre**.

**`SettingWithCopyWarning`**

Aviso de pandas: "estoy haciendo algo ambiguo, puede que tu asignación se pierda". Causado por chaining o asignación sobre subset no-explícito. Solución universal: `.loc[cond, col] = x` en una sola operación.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `SettingWithCopyWarning` aparece y no sé por qué | Estás asignando sobre un resultado de slicing/filter que podría ser vista o copia. **Fix**: `df.loc[mask, 'col'] = valor` (una sola operación) en vez de `df[mask]['col'] = valor`. |
| `df.loc[0:5]` da 6 filas no 5 | loc es **inclusive end**. Para 5 filas: `df.iloc[0:5]` (exclusivo) o `df.loc[0:4]` (inclusivo, manual). |
| `KeyError` con `.loc[5]` cuando hice `set_index('id')` | El index ya no es 0..N — es la columna `id`. **Fix**: usa `.iloc[5]` para posición, o `.loc[<valor_id_real>]` para etiqueta. |
| Asignación a vista no modifica el original | Pandas 3+ va a ser estricto: la vista no se considera modificable. **Fix**: siempre `.loc` para asignar; si necesitas copia, `.copy()` explícito. |
| `df.col1 = x` no actualiza la columna | Como atributo, asignar no agrega columna nueva (lanza UserWarning). **Fix**: `df['col1'] = x` siempre. |

## ❓ Preguntas frecuentes

**❓ ¿`loc` o `iloc`?**

**`loc`** cuando el index es semántico (nombres, fechas, ids). **`iloc`** cuando solo importa la posición (top-K por orden, primeros 10, último). Mezclarlos en el mismo código causa confusión.

**❓ ¿Por qué `loc` slicing es inclusivo?**

Decisión histórica: para labels (strings, fechas), incluir el end es lo intuitivo (`'enero':'marzo'` debe incluir marzo). Para enteros default queda raro — usa `iloc` ahí.

**❓ ¿`at` vale la pena vs `loc` para single value?**

Solo en hot loops (>10k iteraciones). Para uso interactivo, `loc` es lo suficientemente rápido y más legible.

**❓ ¿Cómo selecciono múltiples columnas?**

**Lista entre brackets**: `df[['a', 'b', 'c']]` (devuelve DataFrame). Notar el `[[]]`. Un solo `[]` con string devuelve Series.

**❓ ¿`.loc[mask, cols]` o `.query() + [cols]`?**

Ambos válidos. `.loc[mask, cols]` para máscaras computadas; `.query()` para filtros declarativos largos. Misma velocidad para datasets <100k.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.3
- [pandas Indexing](https://pandas.pydata.org/docs/user_guide/indexing.html)

➡️ **Siguiente:** [024 — Operaciones y alineación](../024-pandas-operaciones-y-alineacion/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Acceso a columna** de 3 formas.

In [ ]:
import numpy as np, pandas as pd

def make_penguins(seed=42, with_na=False):
    """DataFrame sintetico estilo Palmer Penguins (344 filas), sin internet."""
    rng = np.random.default_rng(seed)
    cfg = {  # especie: (n, islas, bill_len, bill_depth, flipper, body_mass)
        'Adelie':    (152, ['Torgersen', 'Biscoe', 'Dream'], 38.8, 18.3, 190, 3700),
        'Chinstrap': (68,  ['Dream'],                        48.8, 18.4, 196, 3733),
        'Gentoo':    (124, ['Biscoe'],                       47.5, 15.0, 217, 5076),
    }
    filas = []
    for sp, (n, islas, bl, bd, fl, bm) in cfg.items():
        for _ in range(n):
            sex = rng.choice(['male', 'female'])
            k = 1.0 if sex == 'male' else 0.93
            filas.append({
                'species': sp,
                'island': rng.choice(islas),
                'bill_length_mm': round(float(rng.normal(bl, 2.5)), 1),
                'bill_depth_mm': round(float(rng.normal(bd, 1.2)), 1),
                'flipper_length_mm': float(round(rng.normal(fl, 6))),
                'body_mass_g': float(round(rng.normal(bm * k, 300))),
                'sex': sex,
            })
    df = pd.DataFrame(filas)
    if with_na:
        idx = rng.choice(df.index, size=12, replace=False)
        df.loc[idx[:6], 'bill_length_mm'] = np.nan
        df.loc[idx[6:], 'sex'] = np.nan
    return df

df = make_penguins()
a = df.species
b = df['species']
c = df.loc[:, 'species']
assert a.equals(b) and b.equals(c)
print('Los 3 metodos devuelven la misma Series. shape:', a.shape)

**Ej. 2 — `loc` inclusivo vs `iloc` exclusivo.**

In [ ]:
print('loc[0:5] ->', df.loc[0:5].shape[0], 'filas (inclusivo -> 6)')
print('iloc[0:5] ->', df.iloc[0:5].shape[0], 'filas (exclusivo -> 5)')
assert df.loc[0:5].shape[0] == 6 and df.iloc[0:5].shape[0] == 5

**Ej. 3 — Filtro compuesto + columnas seleccionadas.**

In [ ]:
sel = df.loc[(df.species == 'Adelie') & (df.sex == 'male') & (df.bill_length_mm > 40),
             ['species', 'island', 'bill_length_mm']]
print(sel.head())
assert (sel.bill_length_mm > 40).all() and list(sel.columns) == ['species', 'island', 'bill_length_mm']

**Ej. 4 — Asignación segura con `.loc`.**

In [ ]:
df.loc[:, 'is_big'] = df['body_mass_g'] > 4500
print(df['is_big'].value_counts())
assert df['is_big'].dtype == bool

**Ej. 5 — SettingWithCopyWarning:** provocar y arreglar.

In [ ]:
import warnings
# Forma que opera sobre un slice (puede disparar el warning segun la version):
sub = df[df.body_mass_g > 4500]
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    sub['flag'] = 1            # se modifica una copia, no el original
# Forma correcta: .loc sobre el DataFrame original, en una sola operacion:
df.loc[df.body_mass_g > 4500, 'flag'] = 1
print('Con .loc la asignacion es explicita y segura. flags:', int(df['flag'].sum()))
assert df['flag'].sum() > 0